# Notebook 03g — Modelos con Tipo de Tumor + Mutaciones Driver (target: SANGRE)

**Objetivo**: igual que 03e pero usando las fusiones teloméricas en **sangre** como target en vez de en tumor.  
**Target**: `log1p(Blood_Telomere fusion rate)` — fusiones detectadas en sangre periférica.  
**Features**:
- `Mixed_def_*` — tipo de tumor en formato binario (58 columnas)
- Un flag binario por cada gen driver con ≥ 2 pacientes (1 = paciente tiene esa mutación, 0 = no)

**¿Por qué sangre?** Detección no invasiva — la biopsia líquida (sangre) es más accesible clínicamente que la muestra de tumor. Si los genes driver también predicen las fusiones en sangre, el modelo tiene aplicación diagnóstica directa.

**Pacientes**: intersección entre `datos_modelo_final.csv` y `Drivers_TF_TVR_TL_to_ALV.xlsx` → ~3885 pacientes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import scipy.stats as stats

import os
os.chdir('..') 

print('Librerías cargadas OK')

## 1. Carga de datos base

In [ ]:
# Cargamos el dataset final que ya tiene el tipo de tumor en formato dummy
df_base = pd.read_csv('datos_modelo_final.csv')
print(f'datos_modelo_final: {df_base.shape}')

# Columnas Mixed_def (tipo de tumor) — ya están calculadas
mixed_cols = [c for c in df_base.columns if c.startswith('Mixed_def_')]
print(f'Columnas tipo de tumor (Mixed_def_*): {len(mixed_cols)}')

# TARGET: fusiones teloméricas en SANGRE (no en tumor como en 03e)
TARGET = 'Blood_Telomere fusion rate (Gt-end/coverage*purity)'

# Nos quedamos solo con ID + target + tipo de tumor
df_base = df_base[['Patient_ID_IF_mod', TARGET] + mixed_cols].copy()
print(f'Columnas seleccionadas: {df_base.shape}')
print(f'Nulos en target: {df_base[TARGET].isna().sum()}')

## 2. Construcción de la matriz de mutaciones

El archivo de mutaciones está en formato **largo** (una fila por mutación).  
Necesitamos convertirlo a formato **ancho**: una columna por gen, con 1 si el paciente tiene esa mutación y 0 si no.

In [ ]:
# Cargamos Sheet1 — formato largo: una fila por mutación
df_drivers = pd.read_excel(
    'Drivers_TF_TVR_TL_to_ALV.xlsx',
    sheet_name='Sheet1',
    header=0,
    usecols=['Patient_ID_IF_mod', 'gene']
)
print(f'Drivers (formato largo): {df_drivers.shape}')
print(f'Pacientes únicos en drivers: {df_drivers["Patient_ID_IF_mod"].nunique()}')
print(f'Genes únicos totales: {df_drivers["gene"].nunique()}')

In [ ]:
# Filtramos genes que aparecen en al menos 2 pacientes
# (igual que 03e — el filtro de >=10 de 03f redujo demasiado los pacientes)
gene_counts = df_drivers.groupby('gene')['Patient_ID_IF_mod'].nunique()
genes_validos = gene_counts[gene_counts >= 2].index.tolist()
print(f'Genes con >= 2 pacientes: {len(genes_validos)}')
print(f'Genes descartados (solo 1 paciente): {df_drivers["gene"].nunique() - len(genes_validos)}')

df_drivers = df_drivers[df_drivers['gene'].isin(genes_validos)].copy()

# Eliminamos duplicados (un paciente puede tener varias mutaciones en el mismo gen)
df_drivers = df_drivers.drop_duplicates(subset=['Patient_ID_IF_mod', 'gene'])

# Pivotamos: filas = pacientes, columnas = genes, valores = 1
df_genes = df_drivers.assign(valor=1).pivot(
    index='Patient_ID_IF_mod',
    columns='gene',
    values='valor'
).fillna(0).astype(int)

# Renombramos columnas para evitar problemas con caracteres especiales
import re
df_genes.columns = ['gene_' + re.sub(r'[^A-Za-z0-9_]', '_', col) for col in df_genes.columns]
df_genes = df_genes.reset_index()

print(f'Matriz de mutaciones: {df_genes.shape}')
print(f'  → {df_genes.shape[0]} pacientes × {df_genes.shape[1]-1} genes')

## 3. Unión de los datasets

In [ ]:
# Unimos: solo los pacientes presentes en ambos datasets
df = df_base.merge(df_genes, on='Patient_ID_IF_mod', how='inner')
print(f'Dataset final (intersección): {df.shape}')
print(f'  → {df.shape[0]} pacientes × {df.shape[1]} columnas')
print(f'Nulos en target: {df[TARGET].isna().sum()}')

## 4. Target — distribución y transformación log1p

El target de sangre tiene distribución muy sesgada pero menos extrema que el tumor (skewness ~4 vs ~58 del tumor). Aun así aplicamos log1p para comprimir los valores altos y estabilizar el entrenamiento.

In [ ]:
y_raw = df[TARGET].copy()

print('=== Target original (sangre) ===')
print(y_raw.describe())
print(f'Skewness: {y_raw.skew():.2f}')
print(f'% ceros exactos: {(y_raw == 0).mean()*100:.1f}%')

# log1p para corregir la asimetría
y = np.log1p(y_raw)
print(f'\nSkewness tras log1p: {y.skew():.2f}')

## 5. Definición de features

In [ ]:
gene_cols = [c for c in df.columns if c.startswith('gene_')]

feature_cols = mixed_cols + gene_cols
X = df[feature_cols].copy()

print(f'Features totales: {X.shape[1]}')
print(f'  - Tipo de tumor (Mixed_def_*): {len(mixed_cols)}')
print(f'  - Genes driver: {len(gene_cols)}')
print(f'Shape final → X: {X.shape} | y: {y.shape}')

## 6. Correlación features vs target

Antes de entrenar los modelos, vemos qué variables tienen más correlación con las **fusiones en sangre**.

In [ ]:
print('Top 15 correlaciones Pearson (features vs target sangre log1p):')
cors = {col: stats.pearsonr(X[col], y) for col in X.columns}
top = sorted(cors.items(), key=lambda x: abs(x[1][0]), reverse=True)[:15]
for col, (r, p) in top:
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    print(f'  {col:45s}  r={r:+.3f}  p={p:.4f}  {sig}')

## 7. Split train / test + escalado

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')

## 8. Función de evaluación

In [ ]:
def evaluar_modelo(nombre, modelo, X_tr, X_te, y_tr, y_te, feature_names):
    modelo.fit(X_tr, y_tr)
    y_pred_log  = modelo.predict(X_te)
    # Deshacer log1p → métricas en espacio original
    y_pred_orig = np.expm1(y_pred_log)
    y_te_orig   = np.expm1(np.array(y_te))

    r2   = r2_score(y_te_orig, y_pred_orig)
    mae  = mean_absolute_error(y_te_orig, y_pred_orig)
    rmse = np.sqrt(mean_squared_error(y_te_orig, y_pred_orig))
    cv   = cross_val_score(modelo, X_tr, y_tr, cv=5, scoring='r2').mean()

    sep = '=' * 55
    print(sep)
    print(f'  {nombre}')
    print(sep)
    print(f'  R²  (test, espacio original):  {r2:.4f}')
    print(f'  R²  (CV-5 train):              {cv:.4f}')
    print(f'  MAE  (espacio original):        {mae:.4f}')
    print(f'  RMSE (espacio original):        {rmse:.4f}')

    coefs = pd.Series(modelo.coef_, index=feature_names)
    nz = coefs[coefs != 0].sort_values(key=abs, ascending=False)
    print(f'  Coefs no nulos: {len(nz)} / {len(coefs)}')
    if len(nz) > 0:
        print('  Top 10 coeficientes:')
        print(nz.head(10).to_string())
    print()

    return {'nombre': nombre, 'modelo': modelo, 'R2_test': r2, 'R2_cv': cv,
            'MAE': mae, 'RMSE': rmse, 'y_pred': y_pred_orig, 'coefs': coefs}

## 9. Lasso — modelo principal para descubrimiento de genes

Lasso pone a cero los genes que no aportan. Los genes con coeficiente distinto de cero son los candidatos a asociarse con la actividad ALT **detectada en sangre** — el dato más relevante clínicamente.

In [ ]:
lasso = Lasso(alpha=0.001, max_iter=10000, random_state=42)
res_lasso = evaluar_modelo(
    'Lasso (alpha=0.001)', lasso,
    X_train_sc, X_test_sc, y_train, y_test,
    X.columns
)

## 10. Ridge

In [ ]:
ridge = Ridge(alpha=1.0, random_state=42)
res_ridge = evaluar_modelo(
    'Ridge (alpha=1.0)', ridge,
    X_train_sc, X_test_sc, y_train, y_test,
    X.columns
)

## 11. Elastic Net

In [ ]:
enet = ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42)
res_enet = evaluar_modelo(
    'Elastic Net (alpha=0.001, l1_ratio=0.5)', enet,
    X_train_sc, X_test_sc, y_train, y_test,
    X.columns
)

## 12. Comparativa final

In [ ]:
resultados = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('modelo', 'y_pred', 'coefs')}
    for r in [res_lasso, res_ridge, res_enet]
]).set_index('nombre')

print('=== COMPARATIVA FINAL (target: sangre) ===')
print(resultados.round(4).to_string())
print()
mejor = resultados['R2_test'].idxmax()
print(f'Mejor modelo (R² test): {mejor}')

# Comparación con 03e (tumor)
print()
print('=== Referencia 03e (mismo modelo, target: TUMOR) ===')
print('  Lasso R²=0.034 | Ridge R²=? | ElasticNet R²=?')
print('  Diferencia clave: target es sangre vs tumor')

## 13. Genes seleccionados por Lasso

Genes que Lasso considera relevantes para predecir las fusiones teloméricas **en sangre**.  
- Coeficiente **positivo** → más mutaciones en ese gen = más fusiones en sangre  
- Coeficiente **negativo** → más mutaciones en ese gen = menos fusiones en sangre

In [ ]:
coefs_lasso = res_lasso['coefs']
genes_seleccionados = coefs_lasso[
    (coefs_lasso != 0) & (coefs_lasso.index.str.startswith('gene_'))
].sort_values(key=abs, ascending=False)

print(f'Genes seleccionados por Lasso: {len(genes_seleccionados)} de {len(gene_cols)}')
print()
print('Genes con coeficiente POSITIVO (más fusiones en sangre):')
pos = genes_seleccionados[genes_seleccionados > 0]
for gene, coef in pos.items():
    nombre = gene.replace('gene_', '')
    print(f'  {nombre:20s}  coef={coef:+.4f}')

print()
print('Genes con coeficiente NEGATIVO (menos fusiones en sangre):')
neg = genes_seleccionados[genes_seleccionados < 0]
for gene, coef in neg.items():
    nombre = gene.replace('gene_', '')
    print(f'  {nombre:20s}  coef={coef:+.4f}')

In [ ]:
# Tipos de tumor seleccionados por Lasso
tumores_seleccionados = coefs_lasso[
    (coefs_lasso != 0) & (coefs_lasso.index.str.startswith('Mixed_def_'))
].sort_values(key=abs, ascending=False)

print(f'Tipos de tumor con coeficiente no nulo: {len(tumores_seleccionados)} de {len(mixed_cols)}')
print()
print(tumores_seleccionados.to_string())

## 14. Comparativa genes: tumor (03e) vs sangre (03g)

¿Los mismos genes predicen las fusiones en tumor y en sangre? Si hay solapamiento, el hallazgo es más robusto.

In [ ]:
# Genes seleccionados en 03e (tumor) — referencia manual de sesión anterior
genes_tumor_03e = {'ATRX', 'CUX1', 'DLG2', 'AGBL4', 'TERT'}

genes_sangre = set(g.replace('gene_', '') for g in genes_seleccionados.index)

solapamiento = genes_tumor_03e & genes_sangre
print(f'Genes en 03e (tumor):  {sorted(genes_tumor_03e)}')
print(f'Genes en 03g (sangre): {sorted(genes_sangre)}')
print(f'Solapamiento:          {sorted(solapamiento) if solapamiento else "ninguno"}')
print()
if solapamiento:
    print('✅ Genes que aparecen en AMBOS targets — hallazgo más robusto:')
    for g in sorted(solapamiento):
        print(f'  - {g}')
else:
    print('⚠️  Sin solapamiento — los predictores de sangre y tumor son distintos')

## 15. Gráficos real vs predicho

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, res in zip(axes, [res_lasso, res_ridge, res_enet]):
    y_pred = np.array(res['y_pred'])
    y_real = np.expm1(np.array(y_test))
    ax.scatter(y_real, y_pred, alpha=0.3, s=10, color='darkorange')
    lim = max(y_real.max(), y_pred.max())
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1, label='Perfecto')
    ax.set_xlabel('Real (espacio original, sangre)')
    ax.set_ylabel('Predicho (espacio original, sangre)')
    ax.set_title(f"{res['nombre']}
R²={res['R2_test']:.4f}")
    ax.legend(fontsize=8)

plt.suptitle('Tipo de tumor + Mutaciones driver — TARGET: SANGRE', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('notebooks/img_03g_sangre_tipo_tumor_mutaciones.png', dpi=150, bbox_inches='tight')
plt.close()
print('Guardado: notebooks/img_03g_sangre_tipo_tumor_mutaciones.png')